## Integrasi Dataset GTFS (Data Logic)

Proyek ini menggunakan empat komponen utama dari standar GTFS untuk membangun graf yang realistis:

stops.txt: Sebagai Nodes (titik pemberhentian).

stop_times.txt: Sebagai Edges (koneksi antar halte berdasarkan jadwal bus).

trips.txt: Sebagai penghubung antara jadwal perjalanan dengan rute spesifik.

routes.txt: Memberikan metadata nama koridor (misal: Koridor 1, Koridor 9).

In [1]:
import pandas as pd
import networkx as nx

# Load Data
stops = pd.read_csv("stops.txt")
stop_times = pd.read_csv("stop_times.txt")
trips = pd.read_csv("trips.txt")
routes = pd.read_csv("routes.txt")

In [2]:
stops.head()

,stop_id,stop_code,stop_name,stop_desc,stop_lat,stop_lon,zone_id,stop_url,location_type,wheelchair_boarding,parent_station,platform_code
0,B00001P,NaN,18 Office Park,NaN,-6.299146,106.83210,1,NaN,0,2,NaN,NaN
1,B00002P,NaN,ABA,NaN,-6.194149,106.83939,1,NaN,0,2,NaN,NaN
2,B00003P,NaN,Acacia Residence,NaN,-6.263358,106.75675,1,NaN,0,2,NaN,NaN
3,B00004P,NaN,ACC Simatupang,NaN,-6.304475,106.84858,1,NaN,0,2,NaN,NaN
4,B00005P,NaN,ACE Hardware,NaN,-6.387532,106.82738,1,NaN,0,2,NaN,NaN


In [3]:
stop_times.head()

,trip_id,stop_sequence,stop_id,arrival_time,departure_time,stop_headsign,pickup_type,drop_off_type,continuous_pickup,continuous_drop_off,shape_dist_traveled,timepoint
0,10A-L02,0,B03282P,05:00:00,05:00:10,Tanjung Priok,0.0,0.0,NaN,NaN,0.000000,0.0
1,10A-L02,1,B03283P,05:03:01,05:03:11,Tanjung Priok,0.0,0.0,NaN,NaN,617.391330,0.0
2,10A-L02,2,B05078P,05:04:48,05:04:58,Tanjung Priok,0.0,0.0,NaN,NaN,969.147805,0.0
3,10A-L02,3,B06039P,05:07:51,05:08:01,Tanjung Priok,0.0,0.0,NaN,NaN,1596.217243,0.0
4,10A-L02,4,B00136P,05:08:46,05:08:56,Tanjung Priok,0.0,0.0,NaN,NaN,1760.710745,0.0


In [4]:
trips.head()

,trip_id,route_id,service_id,trip_headsign,trip_short_name,direction_id,block_id,shape_id,wheelchair_accessible,bikes_allowed
0,6D-L02,6D,SH,Bundaran Senayan dan Stasiun Tebet,Stasiun Tebet - Bundaran Senayan - Stasiun Tebet,0.0,NaN,6D-L02_shp,NaN,NaN
1,6A-L03,6A,SH,Tosari dan Ragunan via Semanggi,Ragunan - Tosari - Ragunan via Semanggi,0.0,NaN,6A-L03_shp,NaN,NaN
2,8-P26,8,SH,Pasar Baru,Damai - Pasar Baru,1.0,NaN,fbqv,NaN,NaN
3,JAK.48A-R02,JAK.48A,SH,Stasiun Tebet,Rusun Karet Tengsin - Stasiun Tebet,1.0,NaN,JAK.48A-R02_shp,NaN,NaN
4,S21-R06,S21,SH,Ciputat,CSW - Ciputat,1.0,NaN,6tw6,NaN,NaN


In [5]:
routes.head()

,route_id,agency_id,route_short_name,route_long_name,route_desc,route_type,route_url,route_color,route_text_color,route_sort_order
0,1,Tije,1,Blok M - Kota,BRT,3,NaN,D62126,FFFFFF,NaN
1,2,Tije,2,Pulo Gadung - Monumen Nasional,BRT,3,NaN,2F489C,FFFFFF,NaN
2,3,Tije,3,Kalideres - Monumen Nasional via Veteran,BRT,3,NaN,FDCB1C,000000,NaN
3,4,Tije,4,Pulo Gadung - Galunggung,BRT,3,NaN,512C62,FFFFFF,NaN
4,5,Tije,5,Kampung Melayu - Ancol,BRT,3,NaN,D46425,000000,NaN


Graf yang dibangun adalah Directed Graph (DiGraph) karena sistem transportasi bersifat asimetris (jalur berangkat belum tentu sama dengan jalur pulang).

In [6]:
# Pembersihan Data (Force String & Strip Whitespace)
stops['stop_id'] = stops['stop_id'].astype(str).str.strip()
stop_times['stop_id'] = stop_times['stop_id'].astype(str).str.strip()
# transfers['from_stop_id'] = transfers['from_stop_id'].astype(str).str.strip()
# transfers['to_stop_id'] = transfers['to_stop_id'].astype(str).str.strip()


# Buat Graf Berarah (DiGraph)
# Karena bus TransJakarta punya arah (A ke B belum tentu sama jalurnya dengan B ke A)
G = nx.DiGraph()

# Menambahkan Nodes (Halte) dengan Atribut Posisi
for _, row in stops.iterrows():
    G.add_node(row['stop_id'], name=row['stop_name'], pos=(row['stop_lon'], row['stop_lat']))

# Tambahkan Edges berdasarkan urutan perjalanan (trip)
stop_times_sorted = stop_times.sort_values(['trip_id', 'stop_sequence'])

# Kita urutkan agar pasangan haltenya benar (n ke n+1)
for trip_id, group in stop_times_sorted.groupby('trip_id'):
    for i in range(len(group)-1):
        u = group.iloc[i]
        v = group.iloc[i+1]

        time_diff = pd.to_timedelta(v['arrival_time']) - pd.to_timedelta(u['departure_time'])
        weight = time_diff.total_seconds()

        G.add_edge(u['stop_id'], v['stop_id'], weight=weight)

print(f"Graf Berhasil Dibangun: {G.number_of_nodes()} node, {G.number_of_edges()} edge")

Graf Berhasil Dibangun: 8946 node, 9901 edge


## Graph Connectivity

In [7]:
# Cek apakah graf terhubung secara lemah (fisik tersambung tanpa liat arah)
is_weakly = nx.is_weakly_connected(G)
print(f"Apakah seluruh jaringan terhubung? {is_weakly}")

num_weakly = nx.number_weakly_connected_components(G)
print(f"Jumlah komponen (pulau) terpisah: {num_weakly}")

# Ambil distribusi ukuran komponen
components = sorted(nx.weakly_connected_components(G), key=len, reverse=True)
print("Ukuran komponen: ", end="")
print(*[len(components[i]) for i in range(0, 10)], "...", sep=", ")

Apakah seluruh jaringan terhubung? False
Jumlah komponen (pulau) terpisah: 1042
Ukuran komponen: 7872, 26, 9, 1, 1, 1, 1, 1, 1, 1, ...


## Visualisasi Graph Nodes/Rute dengan Peta

In [8]:
import plotly.graph_objects as go

# =========================
# EDGES
# Menyiapkan Data Garis (Edges) - Gunakan lat/lon untuk mapbox
# =========================
edge_lat = []
edge_lon = []

for u, v in G.edges():
    if 'pos' in G.nodes[u] and 'pos' in G.nodes[v]:
        lon0, lat0 = G.nodes[u].get('pos')
        lon1, lat1 = G.nodes[v].get('pos')
        edge_lon.extend([lon0, lon1, None])
        edge_lat.extend([lat0, lat1, None])

# =========================
# NODES
# Menyiapkan Data Titik (Nodes)
# =========================
node_lat = []
node_lon = []
node_text = []

for node, data in G.nodes(data=True):
    if 'pos' in data:
        node_lon.append(data['pos'][0])
        node_lat.append(data['pos'][1])
        node_text.append(data.get('name', node))

# =========================
# FIGURE
# Membuat Figure dengan Scattermapbox
# =========================
fig = go.Figure()

# 🔴 Nodes (Halte): Tambahkan Trace Titik Halte
fig.add_trace(go.Scattermapbox(
    lat=node_lat,
    lon=node_lon,
    mode='markers',
    marker=go.scattermapbox.Marker(size=4,color='red',opacity=0.8),
    text=node_text,
    hoverinfo='text',
    name='Halte'
))

# 🔵 Edges (Rute): Tambahkan Trace Garis Rute
fig.add_trace(go.Scattermapbox(
    lat=edge_lat,
    lon=edge_lon,
    mode='lines',
    line=dict(width=1, color='blue'),
    opacity=0.3,
    hoverinfo='none',
    name='Rute'
))

# =========================
# LAYOUT
# Pengaturan Layout Peta
# =========================
fig.update_layout(
    title='Analisis Jaringan TransJakarta',
    mapbox=dict(style="carto-positron",center={"lat": -6.2088, "lon": 106.8456}, zoom=10),
    showlegend=False,
    hovermode='closest',
    width=800,
    height=800,
    margin={"r":0,"t":40,"l":0,"b":0}
)

fig.show()

In [9]:
# Kita ambil komponen terbesar (Weakly Connected)
giant_nodes = max(nx.weakly_connected_components(G), key=len)
G = G.subgraph(giant_nodes).copy()

print(f"Jaringan Utama Berhasil Diisolasi.")
print(f"Jumlah Halte: {G.number_of_nodes()}")
print(f"Jumlah Rute: {G.number_of_edges()}")

Jaringan Utama Berhasil Diisolasi.
Jumlah Halte: 7872
Jumlah Rute: 9865


## Hitung Kepadatan Stasiun

In [10]:
from collections import defaultdict

# Hitung frekuensi kemunculan stop di seluruh trip
arrival_freq = stop_times['stop_id'].value_counts().to_dict()

# Inject ke graph
nx.set_node_attributes(G, arrival_freq, 'arrival_freq')

# Mapping stop_id -> parent_station
stop_to_parent = stops.set_index('stop_id')['parent_station'].to_dict()
stop_to_name = stops.set_index('stop_id')['stop_name'].to_dict()
stop_to_lat = stops.set_index('stop_id')['stop_lat'].to_dict()
stop_to_lon = stops.set_index('stop_id')['stop_lon'].to_dict()

station_stats = defaultdict(lambda: {'total_arrival': 0,'nodes': [],'lat': [],'lon': [],'name': None})

def safe_mean(values):
    values = [v for v in values if v is not None]
    return sum(values) / len(values) if values else None

for node in G.nodes():
    parent = stop_to_parent.get(node)

    # fallback kalau tidak ada parent
    station_id = parent if pd.notna(parent) else node

    freq = G.nodes[node].get('arrival_freq', 0)

    station_stats[station_id]['total_arrival'] += freq
    station_stats[station_id]['nodes'].append(node)
    lat = stop_to_lat.get(node)
    lon = stop_to_lon.get(node)
    if lat is not None and lon is not None:
        station_stats[station_id]['lat'].append(lat)
        station_stats[station_id]['lon'].append(lon)

    if station_stats[station_id]['name'] is None:
        station_stats[station_id]['name'] = stop_to_name.get(node)

station_df = []

for sid, data in station_stats.items():
    station_df.append({
        'station_id': sid,
        'station_name': data['name'],
        'arrival_density': data['total_arrival'],
        'num_stops': len(data['nodes']),
        'lat': safe_mean(data['lat']),
        'lon': safe_mean(data['lon'])
    })

station_df = pd.DataFrame(station_df)

degree_dict = dict(G.degree())

station_df['connection_density'] = station_df['station_id'].apply(
    lambda sid: sum(degree_dict.get(n, 0) for n in station_stats[sid]['nodes'])
)

top_arrival = station_df.sort_values('arrival_density', ascending=False).head(10)

print("Top 10 Stasiun Terpadat (Arrival Density):")
print(top_arrival[['station_name', 'arrival_density']])

Top 10 Stasiun Terpadat (Arrival Density):
              station_name  arrival_density
5869                Juanda               54
5875              Jelambar               50
5870            Pecenongan               48
5886                Petojo               44
6659        Cawang Sentral               44
5876                 Damai               43
6670            Petamburan               42
4626      Monumen Nasional               41
1800  Senayan Bank Jakarta               40
5965                Cawang               40


In [11]:
top_connection = station_df.sort_values('connection_density', ascending=False).head(10)

print("\nTop 10 Transit Hub (Connection Density):")
print(top_connection[['station_name', 'connection_density']])


Top 10 Transit Hub (Connection Density):
              station_name  connection_density
6659        Cawang Sentral                  28
4626      Monumen Nasional                  24
5875              Jelambar                  24
1800  Senayan Bank Jakarta                  24
5938      Mampang Prapatan                  24
6670            Petamburan                  23
5964      Cawang Cililitan                  21
5869                Juanda                  21
5886                Petojo                  20
5292                 CSW 1                  20


In [12]:
# Debug berapa banyak node bermasalah
missing_coords = [
    node for node in G.nodes()
    if stop_to_lat.get(node) is None or stop_to_lon.get(node) is None
]

print(f"Node tanpa koordinat: {len(missing_coords)}")

Node tanpa koordinat: 52


Beberapa node dalam dataset tidak memiliki koordinat geografis, sehingga dilakukan filtering untuk menjaga integritas analisis spasial

## Centrality

Betweeness Centrality membutuhkan 8 menit

In [13]:
# Degree Centrality (lokal)
degree_centrality = nx.degree_centrality(G)

# Betweenness Centrality (global flow)
# betweenness_centrality = nx.betweenness_centrality(G, k=200, seed=42)
betweenness_centrality = nx.betweenness_centrality(G, seed=42)

# Closeness Centrality (aksesibilitas)
closeness_centrality = nx.closeness_centrality(G)

nx.set_node_attributes(G, degree_centrality, 'deg_cent')
nx.set_node_attributes(G, betweenness_centrality, 'bet_cent')
nx.set_node_attributes(G, closeness_centrality, 'close_cent')

In [14]:
from tabulate import tabulate

def aggregate_station_metric(metric_name, agg='mean'):
    result = {}

    for sid, data in station_stats.items():
        values = [G.nodes[n].get(metric_name, 0) for n in data['nodes'] if n in G]

        if not values:
            result[sid] = 0
        else:
            if agg == 'mean':
                result[sid] = sum(values) / len(values)
            elif agg == 'sum':
                result[sid] = sum(values)

    return result

station_df['degree_centrality'] = station_df['station_id'].map(aggregate_station_metric('deg_cent', 'mean'))
station_df['betweenness_centrality'] = station_df['station_id'].map(aggregate_station_metric('bet_cent', 'mean'))
station_df['closeness_centrality'] = station_df['station_id'].map(aggregate_station_metric('close_cent', 'mean'))


top_deg = station_df.sort_values('degree_centrality', ascending=False)[
    ['station_name','degree_centrality']
].head(10).reset_index(drop=True)

top_bet = station_df.sort_values('betweenness_centrality', ascending=False)[
    ['station_name','betweenness_centrality']
].head(10).reset_index(drop=True)

top_clo = station_df.sort_values('closeness_centrality', ascending=False)[
    ['station_name','closeness_centrality']
].head(10).reset_index(drop=True)

combined = pd.DataFrame({
    'Degree (Station)': top_deg['station_name'],
    'Degree': top_deg['degree_centrality'],

    'Betweenness (Station)': top_bet['station_name'],
    'Betweenness': top_bet['betweenness_centrality'],

    'Closeness (Station)': top_clo['station_name'],
    'Closeness': top_clo['closeness_centrality'],
})

combined = combined.round(4)

print(tabulate(combined, headers='keys', tablefmt='fancy_grid', showindex=False))

╒═══════════════════════════╤══════════╤═════════════════════════════╤═══════════════╤═══════════════════════╤═════════════╕
│ Degree (Station)          │   Degree │ Betweenness (Station)       │   Betweenness │ Closeness (Station)   │   Closeness │
╞═══════════════════════════╪══════════╪═════════════════════════════╪═══════════════╪═══════════════════════╪═════════════╡
│ Cibubur Junction          │   0.0019 │ Buperta Cibubur             │        0.2537 │ Buperta Cibubur       │      0.0382 │
├───────────────────────────┼──────────┼─────────────────────────────┼───────────────┼───────────────────────┼─────────────┤
│ Jln. Taman Jatibaru Timur │   0.0019 │ Cibubur Junction            │        0.252  │ Benhil 2              │      0.0378 │
├───────────────────────────┼──────────┼─────────────────────────────┼───────────────┼───────────────────────┼─────────────┤
│ Kejaksaan Agung           │   0.0018 │ Kejaksaan Agung             │        0.107  │ Benhil 3              │      0.0376 │


## Analisis Rute dengan Heatmap Kepadatan

In [15]:
# Load data dengan penanganan tipe data
shapes = pd.read_csv('shapes.txt', low_memory=False)

shapes.head(5)

,shape_id,shape_pt_sequence,shape_pt_lat,shape_pt_lon,shape_dist_traveled
0,6x9c,0,-6.284809,106.904486,0
1,6x9c,1,-6.284823,106.904487,1.560649853
2,6x9c,2,-6.284852,106.904453,6.512448493
3,6x9c,3,-6.284873,106.904429,10.04645344
4,6x9c,4,-6.284899,106.904196,35.96096557


In [16]:
# Copy biar tidak merusak data asli
shapes_clean = shapes.copy()

# Pastikan numerik
shapes_clean['shape_pt_lat'] = pd.to_numeric(shapes_clean['shape_pt_lat'], errors='coerce')
shapes_clean['shape_pt_lon'] = pd.to_numeric(shapes_clean['shape_pt_lon'], errors='coerce')

# Hapus data yang tidak memiliki koordinat valid
shapes_clean = shapes_clean.dropna(subset=['shape_pt_lat', 'shape_pt_lon'])

# Cari Koordinat "Duplikat" (Titik yang dilewati banyak rute)
# Rounding untuk spatial binning (~1 meter)
shapes_clean['lat_bin'] = shapes_clean['shape_pt_lat'].round(5)
shapes_clean['lon_bin'] = shapes_clean['shape_pt_lon'].round(5)

density = (shapes_clean.groupby(['lat_bin', 'lon_bin'])['shape_id'].nunique().reset_index())

density.columns = ['lat', 'lon', 'route_overlap']

In [17]:
import numpy as np
from sklearn.neighbors import BallTree

# Gunakan BallTree untuk Spatial Query (Lebih cepat dari loop)
# Konversi ke radian untuk BallTree
stop_coords = np.deg2rad(stops[['stop_lat', 'stop_lon']].values)
road_coords = np.deg2rad(density[['lat', 'lon']].values)

# Build tree
tree = BallTree(stop_coords, metric='haversine')

# Radius 100 meter
RADIUS = 100 / 6371000  # meter → radian

indices = tree.query_radius(road_coords, r=RADIUS)

stop_load = defaultdict(float)

for i, nearby_stops in enumerate(indices):
    if len(nearby_stops) == 0:
        continue

    load = density.iloc[i]['route_overlap']

    # Distribusi ke semua halte sekitar
    for stop_idx in nearby_stops:
        stop_id = stops.iloc[stop_idx]['stop_id']
        stop_load[stop_id] += load

stop_load_df = pd.DataFrame([{'stop_id': sid, 'physical_load': val} for sid, val in stop_load.items()])
stop_load_df = stop_load_df.merge(stops[['stop_id', 'stop_name', 'parent_station', 'stop_lat', 'stop_lon']], on='stop_id', how='left')

station_load = (stop_load_df.groupby('parent_station', dropna=False).agg(
    {'stop_name': 'first', 'physical_load': 'sum', 'stop_lat': 'mean', 'stop_lon': 'mean'}
    ).reset_index().dropna(subset=['parent_station']).sort_values('physical_load', ascending=False))

print("Top 20 Halte dengan Beban Lalu Lintas Tertinggi:")
print(station_load[['stop_name', 'parent_station', 'physical_load']].head(20), "\n")

Top 20 Halte dengan Beban Lalu Lintas Tertinggi:
                          stop_name parent_station  physical_load
31           Cawang Sentral Akses D        H00030P         4275.0
94           Kampung Melayu Akses A        H00095P         3140.0
42                            CSW 1        H00041P         2398.0
210     Senen TOYOTA Rangga Akses B        H00212P         2155.0
82                 Jelambar Akses B        H00083P         2100.0
79   Jak Lingko Tanah Abang Akses B        H00079P         2051.0
190              Rawa Barat Akses B        H00192P         1790.0
91                           Juanda        H00092P         1777.0
71                 Grogol Reformasi        H00071P         1646.0
254                          Velbak        H00257P         1563.0
183             Pulo Gadung Akses A        H00184P         1554.0
2                    Blok M Jalur 2        B06577P         1518.0
237                   Tanjung Priok        H00240P         1452.0
273                  Cawang

In [18]:
import plotly.graph_objects as go

fig = go.Figure(go.Densitymapbox(
    lat=density['lat'],
    lon=density['lon'],
    z=density['route_overlap'],
    radius=4,
    colorscale=[
        [0.0, "green"],
        [0.2, "yellow"],
        [1.0, "red"]
    ],
    zmin=1,
    zmax=density['route_overlap'].quantile(0.95),  # hindari outlier ekstrem
    colorbar=dict(title="Route Overlap")
))

fig.update_layout(
    title='Heatmap Kepadatan Rute TransJakarta',
    mapbox=dict(
        style="carto-positron",
        center={"lat": -6.2088, "lon": 106.8456},
        zoom=11
    ),
    margin={"r":0,"t":80,"l":0,"b":0},
    height=800
)

fig.show()

In [19]:
import pandas as pd
import numpy as np

analysis = []

for node, data in G.nodes(data=True):
    analysis.append({
        'stop_id': node,
        'stop_name': data.get('name'),
        'arrival_freq': data.get('arrival_freq', 0),
        'betweenness': data.get('bet_cent', 0),
        'degree': data.get('deg_cent', 0),
        'closeness': data.get('close_cent', 0)
    })

analysis_df = pd.DataFrame(analysis)

def normalize(col):
    return (col - col.min()) / (col.max() - col.min() + 1e-9)

# # Normalisasi agar bisa dibandingkan (0-1)
analysis_df['norm_arrival'] = normalize(analysis_df['arrival_freq'])
analysis_df['norm_betweenness'] = normalize(analysis_df['betweenness'])
analysis_df['norm_degree'] = normalize(analysis_df['degree'])
analysis_df['norm_closeness'] = normalize(analysis_df['closeness'])
analysis_df['importance'] = (
    0.5 * analysis_df['norm_betweenness'] +
    0.3 * analysis_df['norm_closeness'] +
    0.2 * analysis_df['norm_degree']
)
analysis_df['service_gap'] = analysis_df['importance'] - analysis_df['norm_arrival']

top_gap = analysis_df.sort_values('service_gap', ascending=False).head(15)

# # Tampilkan Halte paling "Terabaikan" oleh sistem
print("Top Halte 'Under-Served' (Penting tapi Kurang Dilayani):")
print(top_gap[['stop_name', 'service_gap', 'importance', 'norm_arrival']])
analysis_df['service_gap'] = analysis_df['importance'] - analysis_df['norm_arrival']

Top Halte 'Under-Served' (Penting tapi Kurang Dilayani):
                       stop_name  service_gap  importance  norm_arrival
233              Buperta Cibubur     0.571428    0.942857      0.371429
6244              Simpang Cawang     0.386733    0.501018      0.114286
1358       Jln. Sungai Sambas VI     0.347356    0.404499      0.057143
6338            JIEXPO Kemayoran     0.338190    0.395333      0.057143
6241              Simpang Cawang     0.335320    0.535320      0.200000
7200            Grogol Reformasi     0.334623    0.391766      0.057143
4943     Kantor Walikota Jakut 1     0.320481    0.434767      0.114286
788              Jln. Bulungan 2     0.319320    0.347891      0.028571
4781  Tanah Pasir Gedong Panjang     0.315660    0.401374      0.085714
8       Ahmad Yani Pisangan Baru     0.311714    0.397429      0.085714
1711                    Komdak 2     0.306885    0.506885      0.200000
4792              Tegal Parang 1     0.304894    0.447751      0.142857
7526   

In [20]:
import math
import random

def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # km
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)

    a = (math.sin(dlat/2)**2 +
         math.cos(math.radians(lat1)) *
         math.cos(math.radians(lat2)) *
         math.sin(dlon/2)**2)

    return 2 * R * math.asin(math.sqrt(a))


for u, v in G.edges():
    pos_u = G.nodes[u].get('pos')
    pos_v = G.nodes[v].get('pos')

    if pos_u is None or pos_v is None:
        continue  # skip edge tanpa koordinat

    lon1, lat1 = pos_u
    lon2, lat2 = pos_v

    dist = haversine(lat1, lon1, lat2, lon2)
    G[u][v]['distance'] = dist

In [21]:
def analyze_route(G, source, target):
    try:
        path = nx.shortest_path(G, source=source, target=target, weight='distance')

        # total jarak rute
        total_dist = sum(
            G[path[i]][path[i+1]]['distance']
            for i in range(len(path)-1)
        )

        # jarak lurus
        lat1, lon1 = G.nodes[source]['pos'][1], G.nodes[source]['pos'][0]
        lat2, lon2 = G.nodes[target]['pos'][1], G.nodes[target]['pos'][0]

        direct_dist = haversine(lat1, lon1, lat2, lon2)

        sinuosity = total_dist / direct_dist if direct_dist > 0 else 1

        return {
            'stops': len(path),
            'distance': total_dist,
            'sinuosity': sinuosity
        }

    except:
        return None

def evaluate_network(G, samples=100):
    nodes = list(G.nodes())
    results = []

    for _ in range(samples):
        u, v = random.sample(nodes, 2)
        res = analyze_route(G, u, v)

        if res:
            results.append(res)

    return pd.DataFrame(results)

results_df = evaluate_network(G, samples=150)

print("=== NETWORK PERFORMANCE ===")
print(f"Avg Stops      : {results_df['stops'].mean():.2f}")
print(f"Avg Distance   : {results_df['distance'].mean():.2f} km")
print(f"Avg Sinuosity  : {results_df['sinuosity'].mean():.2f}")

inefficient = results_df.sort_values('sinuosity', ascending=False).head(10)

print("\nRute Paling Tidak Efisien:")
print(inefficient)

=== NETWORK PERFORMANCE ===
Avg Stops      : 59.44
Avg Distance   : 22.89 km
Avg Sinuosity  : 1.97

Rute Paling Tidak Efisien:
     stops   distance  sinuosity
50      25   7.338222   7.573295
142     68  23.905452   6.813037
8       39  11.300984   5.769115
19      11   2.959711   4.976621
25      20   7.234499   4.933358
140     36   9.514481   4.636434
24      83  20.146204   4.232626
76      81  28.742980   4.090113
135     37  38.142097   3.879921
12      42  15.684140   3.548241


Sinuosity Index Analysis: Jika rata-rata indeks adalah 1.2 - 1.4, artinya
jaringan TransJakarta cukup efisien (jalur bus mengikuti arah tujuan dengan baik). Jika di atas 1.5, berarti rute-rutenya terlalu banyak memutar (tidak efisien secara jarak).

Analisis ini menunjukkan bahwa graf yang dibuat bukan sekadar gambar, tapi merupakan model simulasi yang bisa memprediksi pengalaman penumpang di lapangan.

## Network Efficiency & Robustness Analysis
Seberapa efisien jaringan transportasi secara keseluruhan

In [22]:
efficiency = nx.global_efficiency(G.to_undirected())
print(efficiency)

0.043593782122101056


Simulasi hapus node dengan centrality tinggi dan melihat dampaknya

In [23]:
G_copy = G.copy()
critical_node = max(betweenness_centrality, key=betweenness_centrality.get)

G_copy.remove_node(critical_node)

new_efficiency = nx.global_efficiency(G_copy.to_undirected())
print(new_efficiency)

0.04320674139711678
